In [ ]:
!pip install opencv-python pandas tensorflow tensorflow-hub transformers accelerate transformers torch sentencepiece sacremoses

In [ ]:
import os
import io
import glob
import json
import shutil
import zipfile
import requests
import pandas as pd
import csv
import cv2
import re
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os
import json
import glob
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
def find_and_check_csv():
    if os.path.exists('/kaggle/working'):
        print("Đang chạy trên môi trường Kaggle...")
        csv_path = '/kaggle/input/datasets/dipthnnguyn/bactch1/Batch1.csv' 
        output_dir = '/kaggle/working/dataset_unzipped'
    else:
        print("Đang chạy trên máy tính cá nhân (Local)...")
        csv_path = './Batch1.csv' 
        output_dir = './dataset_unzipped'
    os.makedirs(output_dir, exist_ok=True)
    try:
        df = pd.read_csv(csv_path)
        print(f"Đã nạp file CSV thành công. Tổng số dòng: {len(df)}")
        # In ra danh sách các cột để bạn đối chiếu xem đã gõ đúng tên cột chứa link video chưa
        print(f"Các cột hiện có trong file CSV: {list(df.columns)}")
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file CSV tại {csv_path}")
        df = pd.DataFrame() 
    return output_dir

In [ ]:
def _extract_keyframe_info(img_path: str, fallback_index: int, video_id: str):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    numbers = re.findall(r'\d+', stem)

    if len(numbers) >= 2:
        keyframe_id = numbers[-2].zfill(4)
        frame_index = int(numbers[-1])
    elif len(numbers) == 1:
        keyframe_id = numbers[0].zfill(4)
        frame_index = int(fallback_index)
    else:
        keyframe_id = f"{fallback_index:04d}"
        frame_index = int(fallback_index)

    return str(video_id), keyframe_id, frame_index


def keyframes_to_btc_json_per_frame(
    keyframe_paths, 
    output_folder, 
    model, 
    start_time=None, 
    max_hours=5, 
    source_dir_to_delete=None
):
    """
    Thêm các tham số:
    - start_time: Thời điểm bắt đầu chạy toàn bộ pipeline (time.time())
    - max_hours: Số giờ tối đa được phép chạy
    - source_dir_to_delete: Thư mục chứa video/ảnh gốc cần xóa nếu quá thời gian
    """
    
    # Nếu không truyền start_time từ ngoài vào, lấy thời điểm gọi hàm làm mốc
    if start_time is None:
        start_time = time.time()

    video_id = os.path.basename(
        os.path.normpath(output_folder)
    )

    # --- NEW CODE ADDED HERE ---
    # List of folders to ignore extracted from image_a38bca.png
    allowed_prefixes = (
        "L23", 
        "L24"
    )
    if not video_id.startswith(allowed_prefixes):
        print(f"Skipping folder: {video_id} (not in allowed list)")
        return
    # ---------------------------

    os.makedirs(output_folder, exist_ok=True)
    keyframe_paths = sorted(keyframe_paths)
    CONF_THRESHOLD = 0.05
    
    for idx, img_path in enumerate(keyframe_paths, start=1):
        # === KIỂM TRA THỜI GIAN VÀ XÓA FILE NẾU QUÁ HẠN ===
        elapsed_hours = (time.time() - start_time) / 3600
        if elapsed_hours >= max_hours:
            print(f"\n[TIMEOUT GUARD] Đạt mốc {elapsed_hours:.2f}h tại frame {idx}. Đang dừng an toàn...")
            
            # Ưu tiên xóa thư mục được chỉ định, nếu không thì xóa thư mục cha của ảnh hiện tại
            dir_to_clean = source_dir_to_delete if source_dir_to_delete else os.path.dirname(img_path)
            
            if os.path.exists(dir_to_clean):
                shutil.rmtree(dir_to_clean, ignore_errors=True)
                print(f" -> Đã xóa thư mục video/ảnh chưa xử lý xong: {dir_to_clean}")
                
            print("Dừng chương trình.")
            sys.exit(0)
        # ===================================================

        frame = cv2.imread(img_path)
        if frame is None:
            print(f"Không đọc được ảnh: {img_path}")
            continue
        
        video_id, keyframe_id, frame_index = _extract_keyframe_info(img_path, idx, video_id)
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = tf.convert_to_tensor(img_rgb, dtype=tf.float32)
        img_tensor = img_tensor / 255.0
        img_tensor = tf.expand_dims(img_tensor, axis=0)
        
        results = model(img_tensor)
        boxes = results['detection_boxes'].numpy()
        scores = results['detection_scores'].numpy()
        entities = results['detection_class_entities'].numpy()
        
        if boxes.ndim == 3:
            boxes = boxes[0]
        scores = np.asarray(scores).reshape(-1)
        entities = np.asarray(entities).reshape(-1)
        boxes = np.asarray(boxes)
        
        if boxes.ndim == 1:
            boxes = boxes.reshape(1, -1)
            
        num_objects = min(len(scores), len(boxes), len(entities))
        objects = []
        
        for i in range(num_objects):
            score = float(scores[i])
            if score < CONF_THRESHOLD:
                continue
                
            ymin, xmin, ymax, xmax = boxes[i]
            label = entities[i]
            
            if isinstance(label, bytes):
                label = label.decode('utf-8', errors='ignore')
            else:
                label = str(label)

            objects.append({
                "label": label,
                "score": score,
                "bbox": [
                    float(ymin),
                    float(xmin),
                    float(ymax),
                    float(xmax)
                ]
            })
            
        objects.sort(
            key=lambda obj: (
                obj["bbox"][0],
                obj["bbox"][1]
            )
        )
        
        results_dict = {
            "video_id": video_id,
            "keyframe_id": keyframe_id,
            "objects": objects
        }
        
        output_json_path = os.path.join(
            output_folder,
            f"{keyframe_id}.json"
        )
        
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(
                results_dict,
                f,
                ensure_ascii=False,
                indent=2
            )
        print(f"  -> Saved: {output_json_path} ({len(objects)} objects)")

In [ ]:
def translate_json_labels_from_file(in_dir: str, json_dir: str):
    # Đường dẫn tới file json chứa từ điển dịch theo yêu cầu
    dict_path = '/kaggle/input/datasets/dipthnnguyn/translated-rcnn/english_vietnamese.json'
    
    print(f"1. Đang tải từ điển dịch từ file: {dict_path}...")
    try:
        with open(dict_path, 'r', encoding='utf-8') as f:
            dict_list = json.load(f)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file từ điển tại {dict_path}. Vui lòng kiểm tra lại đường dẫn!")
        return
    except json.JSONDecodeError:
        print("Lỗi: File từ điển không đúng định dạng JSON.")
        return
    
    # Chuyển đổi định dạng List các Dictionary sang dạng Key: Value mapping để tra cứu nhanh (O(1))
    translation_cache = {item['English']: item['Vietnamese'] for item in dict_list}
    print(f" -> Đã nạp thành công {len(translation_cache)} từ vựng tiếng Anh.")

    print(f"2. Đang quét các file JSON trong thư mục: {in_dir}...")
    json_files = glob.glob(os.path.join(in_dir, "**", "*.json"), recursive=True)
    
    if not json_files:
        print("Không tìm thấy file JSON nào!")
        return
    
    print(f"3. Đang cập nhật nhãn tiếng Việt và lưu sang thư mục: {json_dir}...")
    
    # Tạo thư mục gốc chứa output nếu chưa có
    os.makedirs(json_dir, exist_ok=True)
    
    # Sửa in_files thành json_files
    for filepath in json_files:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Duyệt qua các đối tượng 'objects' trong file JSON
        for obj in data.get('objects', []):
            eng_label = obj.get('label')
            
            if not eng_label:
                continue
                
            # Tra cứu nhãn tiếng Anh (ưu tiên kiểm tra chính xác, sau đó kiểm tra dạng Capitalize)
            if eng_label in translation_cache:
                obj['label'] = translation_cache[eng_label]
            elif eng_label.capitalize() in translation_cache:
                obj['label'] = translation_cache[eng_label.capitalize()]
        
        # Tính toán đường dẫn lưu file mới để giữ nguyên cấu trúc thư mục con (nếu có)
        rel_path = os.path.relpath(filepath, in_dir)
        out_filepath = os.path.join(json_dir, rel_path)
        
        # Đảm bảo thư mục con chứa file output tồn tại
        os.makedirs(os.path.dirname(out_filepath), exist_ok=True)
        
        # Lưu file vào thư mục mới (Lưu tất cả các file để thư mục mới có đủ bộ dataset)
        with open(out_filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
                
    print("=== HOÀN TẤT DỊCH TỪ FILE DICTIONARY ===")

In [ ]:
def process_local_datasets(dataset_dirs, json_dir):
    print(" - Đang nạp mô hình Faster R-CNN Inception-ResNet-v2 (TF Hub)...")
    print("   (Quá trình này có thể mất vài phút nếu tải lần đầu)")
    module_handle = "https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1"
    model = hub.load(module_handle).signatures['default']
    
    os.makedirs(json_dir, exist_ok=True)

    for ds_path in dataset_dirs:
        print(f"\n=== BẮT ĐẦU QUÉT DATASET: {ds_path} ===")
        if not os.path.exists(ds_path):
            print(f"Lỗi: Không tìm thấy thư mục {ds_path}")
            continue

        # Tìm tất cả các file ảnh trong toàn bộ thư mục dataset
        image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')
        keyframe_files = []
        for ext in image_extensions:
            keyframe_files.extend(glob.glob(os.path.join(ds_path, '**', ext), recursive=True))
        
        if not keyframe_files:
            print(f" - Không tìm thấy file ảnh nào trong {ds_path}.")
            continue
        
        # SỬA LỖI Ở ĐÂY: Gom nhóm ảnh theo đường dẫn thư mục tương đối để giữ nguyên cấu trúc
        frames_by_dir = {}
        for img_path in keyframe_files:
            # Lấy đường dẫn thư mục tương đối (VD: keyframes_batch_1/keyframes/L21_V001)
            rel_dir = os.path.relpath(os.path.dirname(img_path), ds_path)
            
            if rel_dir not in frames_by_dir:
                frames_by_dir[rel_dir] = []
            frames_by_dir[rel_dir].append(img_path)

        # Xử lý từng thư mục giữ nguyên cấu trúc
        for rel_dir, img_list in frames_by_dir.items():
            # Tạo đường dẫn output khớp với cấu trúc thư mục gốc
            video_json_folder = os.path.join(json_dir, rel_dir)
            
            print(f" * Đang chạy Object Detection cho nhóm: {rel_dir} ({len(img_list)} ảnh)")
            keyframes_to_btc_json_per_frame(img_list, video_json_folder, model)

def start_pipeline():
    print("Khởi động hệ thống xử lý Keyframes trực tiếp từ Kaggle Datasets...")
    
    # Cập nhật đường dẫn dataset chính xác theo cấu trúc của bạn
    dataset_dirs = [
        '/kaggle/input/datasets/keyframes' # Sửa tên thư mục dataset của bạn vào đây
    ]
    
    OUTPUT_JSON_DIR = '/kaggle/working/json_results'
    
    # Chạy trích xuất object
    process_local_datasets(dataset_dirs, OUTPUT_JSON_DIR)

# Khởi chạy
start_pipeline()